# 2PC Perf
调试2PC(CHEETAH)协议中的参数，对比性能

1. 128in, 8out
2. 256in, 256out
3. NLP precision

## (1) 初始化
### 1.1 加载模型和权重

In [1]:
import jax


from flax_rnn.helper import (
    load_from_cache, prefill, step_fn,
    sampler_baseline, sampler_greedy, sampler_min_p, sampler_top_k, sampler_legacy,
    generate_topk, generate_greedy, generate_minp,
    generate, generate_demo,
)

# base_path = "/root/.cache/huggingface/hub/models--state-spaces--mamba-130m-hf/snapshots/1e76775f628fbf1350fbe4dbb3d971ba64af25a1"
base_path = "/root/shared-nvme/hf_cache/hub/models--state-spaces--mamba-130m-hf/snapshots/1e76775f628fbf1350fbe4dbb3d971ba64af25a1"
model, params, tokenizer = load_from_cache(base_path)

print("model loaded")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


model loaded


In [2]:
# 验证模型加载
generate_demo(model, params, tokenizer, "python is")

python is in a subdirectory, and the directory has a subdirectory, named, for instance,.pyx.

I have a project that uses Python, and I'd like to build a subdirectory to use the code from the project, without

### 1.2 定义SPU模拟器

In [3]:
import sml.utils.emulation as emulation

mode = emulation.Mode.MULTIPROCESS

# note: in MULTIPROCESS mode, bandwidth and latency doesn't work
# emulation.CLUSTER_ABY3_3PC is a hard-coded string
# we copied it to current folder
emulator = emulation.Emulator(
    "2pc.json",
    mode,
    bandwidth=100,
    latency=10
)

emulator.up()

[2026-04-27 13:33:15,519]-[INFO]-[emulation.py:112]: Start multiprocess cluster...
[2026-04-27 13:33:16,080] [ForkServerProcess-1] Starting grpc server at 127.0.0.1:61920
[2026-04-27 13:33:16,092] [ForkServerProcess-5] Starting grpc server at 127.0.0.1:61924
[2026-04-27 13:33:16,107] [ForkServerProcess-3] Starting grpc server at 127.0.0.1:61922
[2026-04-27 13:33:16,136] [ForkServerProcess-2] Starting grpc server at 127.0.0.1:61921
[2026-04-27 13:33:16,140] [ForkServerProcess-4] Starting grpc server at 127.0.0.1:61923
[2026-04-27 13:33:17,594] [ForkServerProcess-1] Run : builtin_spu_init at node:0
[2026-04-27 13:33:17,594] [ForkServerProcess-2] Run : builtin_spu_init at node:1
[2026-04-27 13:33:17,594] [ForkServerProcess-3] Run : builtin_spu_init at node:2
I0427 13:33:17.603651 1743831     0 external/brpc~/src/brpc/server.cpp:1195] Server[yacl::link::transport::internal::ReceiverServiceImpl] is serving on port=61930.
W0427 13:33:17.603668 1743831     0 external/brpc~/src/brpc/server.cpp

### 1.3 初始化测试数据

In [4]:
prompt = "python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
logits, _ = prefill(model, params, input_ids)

s_logits = emulator.seal(logits)
s_params, s_input_ids = emulator.seal(params, input_ids)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
[2026-04-27 13:33:17,736] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-27 13:33:19,186] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-27 13:33:19,268] [ForkServerProcess-4] Run : <lambda> at node:3


## (2) 采样性能评估
### 2.1 明文测试采样函数

- 基线：什么都不干，输入logits，返回0，作为基础通信量和时间的参考
- 贪心采样：取原始值最高的logit输出，无需softmax等计算，V-1次比较
- topk采样：取原始值最高的k个，在这k个中取样，O(kV)
- minp采样：找到最高概率pmax，设置最低概率pmin=minp*pmax，在概率pmin-pmax的logits中采样
- 老代码：跑出来非常糟糕结果的老采样代码

In [5]:
next_id = sampler_baseline(logits)
print(prompt, tokenizer.decode(next_id), sep="")

next_id =sampler_top_k(logits)
print(prompt, tokenizer.decode(next_id), sep="")

next_id =sampler_min_p(logits)
print(prompt, tokenizer.decode(next_id), sep="")

next_id =sampler_legacy(logits)
print(prompt, tokenizer.decode(next_id), sep="")

python is<|endoftext|>
python is not
python is not
python is in


### 2.2 spu baseline

In [6]:
next_id = emulator.run(sampler_baseline)(s_logits)

[2026-04-27 13:33:19.551] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-27 13:33:19.593] [info] [api.cc:172] [Profiling] SPU execution sampler_baseline completed, input processing took 6.9e-07s, execution took 0.000708208s, output processing took 1.42e-06s, total time 0.000710318s.
[2026-04-27 13:33:19.593] [info] [api.cc:220] HLO profiling: total time 1.479e-05
[2026-04-27 13:33:19.593] [info] [api.cc:223] - pphlo.constant, executed 1 times, duration 1.479e-05s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-27 13:33:19.593] [info] [api.cc:220] HAL profiling: total time 0
[2026-04-27 13:33:19.593] [info] [api.cc:220] MPC profiling: total time 0
[2026-04-27 13:33:19.593] [info] [api.cc:233] Link details: total send bytes 0, recv bytes 0, send actions 0, recv actions 0


[2026-04-27 13:33:19,550] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 13:33:19,568] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 13:33:19,581] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-27 13:33:19,581] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-27 13:33:19,581] [ForkServerProcess-3] Run : builtin_spu_run at node:2
[2026-04-27 13:33:19,583] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-27 13:33:19,584] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-27 13:33:19,588] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-27 13:33:19,597] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-27 13:33:19,598] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0
[2026-04-27 13:33:19,599] [ForkServerProcess-2] RunR: builtin_fetch_object at node:1
[2026-04-27 13:33:19,601] [ForkServerProcess-3] RunR: builtin_fetch_object at node:2


In [7]:
print(prompt, tokenizer.decode(next_id), sep="")

python is<|endoftext|>


### 2.3 spu greedy

In [8]:
next_id = emulator.run(sampler_greedy)(s_logits)

[2026-04-27 13:33:19.655] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-27 13:33:19.656] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-27 13:33:19.659] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95


[2026-04-27 13:33:19,619] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 13:33:19,629] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 13:33:19,648] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-27 13:33:19,648] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-27 13:33:19,648] [ForkServerProcess-3] Run : builtin_spu_run at node:2
[2026-04-27 13:33:19,649] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-27 13:33:19,651] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-27 13:33:19,652] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3


[2026-04-27 13:33:20.104] [info] [api.cc:172] [Profiling] SPU execution sampler_greedy completed, input processing took 7.8e-07s, execution took 0.448661449s, output processing took 1.1e-06s, total time 0.448663329s.
[2026-04-27 13:33:20.104] [info] [api.cc:220] HLO profiling: total time 0.447741082
[2026-04-27 13:33:20.104] [info] [api.cc:223] - pphlo.reduce, executed 1 times, duration 0.442235632s, send bytes 26236313 recv bytes 19131385, send actions 842, recv actions 848
[2026-04-27 13:33:20.104] [info] [api.cc:223] - pphlo.iota, executed 1 times, duration 0.004524631s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-27 13:33:20.104] [info] [api.cc:223] - pphlo.convert, executed 3 times, duration 0.000935449s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-27 13:33:20.104] [info] [api.cc:223] - pphlo.reshape, executed 1 times, duration 2.806e-05s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-27 13:33:20.104] [info] [api.c

[2026-04-27 13:33:20,107] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-27 13:33:20,108] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0
[2026-04-27 13:33:20,109] [ForkServerProcess-2] RunR: builtin_fetch_object at node:1
[2026-04-27 13:33:20,110] [ForkServerProcess-3] RunR: builtin_fetch_object at node:2


In [9]:
print(prompt, tokenizer.decode(next_id), sep="")

python is not


### 2.4 spu top k

In [10]:
next_id = emulator.run(sampler_top_k)(s_logits)

[2026-04-27 13:33:20,161] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 13:33:20,171] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 13:33:20,245] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-27 13:33:20,245] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-27 13:33:20,245] [ForkServerProcess-3] Run : builtin_spu_run at node:2
[2026-04-27 13:33:20,246] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-27 13:33:20,249] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-27 13:33:20,251] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3


[2026-04-27 13:33:21.037] [info] [api.cc:172] [Profiling] SPU execution sampler_top_k completed, input processing took 1.07e-06s, execution took 0.784646162s, output processing took 1.24e-06s, total time 0.784648472s.
[2026-04-27 13:33:21.037] [info] [api.cc:220] HLO profiling: total time 0.7824727960000001
[2026-04-27 13:33:21.037] [info] [api.cc:223] - pphlo.reduce, executed 1 times, duration 0.431779694s, send bytes 21981497 recv bytes 30679209, send actions 842, recv actions 848
[2026-04-27 13:33:21.037] [info] [api.cc:223] - pphlo.custom_call: mhlo.topk, executed 1 times, duration 0.162579667s, send bytes 5668566 recv bytes 4864086, send actions 151, recv actions 150
[2026-04-27 13:33:21.037] [info] [api.cc:223] - pphlo.less, executed 2 times, duration 0.09244536s, send bytes 3419040 recv bytes 3419040, send actions 9, recv actions 9
[2026-04-27 13:33:21.037] [info] [api.cc:223] - pphlo.log, executed 2 times, duration 0.019689509s, send bytes 0 recv bytes 0, send actions 0, recv a

[2026-04-27 13:33:21,039] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-27 13:33:21,040] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0
[2026-04-27 13:33:21,042] [ForkServerProcess-2] RunR: builtin_fetch_object at node:1
[2026-04-27 13:33:21,043] [ForkServerProcess-3] RunR: builtin_fetch_object at node:2


In [11]:
print(prompt, tokenizer.decode(next_id), sep="")

python is not


### 2.5 spu min p

In [12]:
next_id = emulator.run(sampler_min_p)(s_logits)

[2026-04-27 13:33:21,095] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 13:33:21,105] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 13:33:21,181] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-27 13:33:21,181] [ForkServerProcess-3] Run : builtin_spu_run at node:2
[2026-04-27 13:33:21,181] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-27 13:33:21,183] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-27 13:33:21,183] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-27 13:33:21,186] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3


[2026-04-27 13:33:21.951] [info] [api.cc:172] [Profiling] SPU execution sampler_min_p completed, input processing took 6.5e-07s, execution took 0.762724929s, output processing took 1.38e-06s, total time 0.762726959s.
[2026-04-27 13:33:21.952] [info] [api.cc:220] HLO profiling: total time 0.7604309749999999
[2026-04-27 13:33:21.952] [info] [api.cc:223] - pphlo.reduce, executed 2 times, duration 0.574969949s, send bytes 31598597 recv bytes 31445477, send actions 1089, recv actions 1041
[2026-04-27 13:33:21.952] [info] [api.cc:223] - pphlo.less, executed 2 times, duration 0.091170121s, send bytes 3419040 recv bytes 3419040, send actions 9, recv actions 9
[2026-04-27 13:33:21.952] [info] [api.cc:223] - pphlo.log, executed 2 times, duration 0.019297429s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-27 13:33:21.952] [info] [api.cc:223] - pphlo.or, executed 21 times, duration 0.018030079s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-27 13:33:21.952

[2026-04-27 13:33:21,955] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-27 13:33:21,956] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0
[2026-04-27 13:33:21,957] [ForkServerProcess-2] RunR: builtin_fetch_object at node:1
[2026-04-27 13:33:21,958] [ForkServerProcess-3] RunR: builtin_fetch_object at node:2


In [13]:
print(prompt, tokenizer.decode(next_id), sep="")

python is not


### 2.6 spu legacy

In [14]:
next_id = emulator.run(sampler_legacy)(s_logits)

[2026-04-27 13:33:22,018] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 13:33:22,028] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 13:33:22,151] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-27 13:33:22,152] [ForkServerProcess-3] Run : builtin_spu_run at node:2
[2026-04-27 13:33:22,152] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-27 13:33:22,154] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-27 13:33:22,154] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-27 13:33:22,156] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3


[2026-04-27 13:33:25.540] [info] [api.cc:172] [Profiling] SPU execution sampler_legacy completed, input processing took 1.09e-06s, execution took 3.380975898s, output processing took 1.18e-06s, total time 3.380978168s.
[2026-04-27 13:33:25.540] [info] [api.cc:220] HLO profiling: total time 3.3773132550000002
[2026-04-27 13:33:25.540] [info] [api.cc:223] - pphlo.log, executed 3 times, duration 1.334754579s, send bytes 135554880 recv bytes 76023360, send actions 43, recv actions 42
[2026-04-27 13:33:25.540] [info] [api.cc:223] - pphlo.exponential, executed 1 times, duration 0.562145712s, send bytes 62548320 recv bytes 54905760, send actions 77, recv actions 69
[2026-04-27 13:33:25.540] [info] [api.cc:223] - pphlo.reduce, executed 3 times, duration 0.542808733s, send bytes 36684229 recv bytes 18351109, send actions 1085, recv actions 1041
[2026-04-27 13:33:25.540] [info] [api.cc:223] - pphlo.divide, executed 1 times, duration 0.453013119s, send bytes 54905760 recv bytes 54503520, send act

[2026-04-27 13:33:25,543] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-27 13:33:25,544] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0
[2026-04-27 13:33:25,545] [ForkServerProcess-2] RunR: builtin_fetch_object at node:1
[2026-04-27 13:33:25,546] [ForkServerProcess-3] RunR: builtin_fetch_object at node:2


In [15]:
print(prompt, tokenizer.decode(next_id), sep="")

python is not


## (3) 验证推理性能
### 3.1 定义运行函数

In [16]:
# 目的是规避model参数，给函数加上jit
# 重要：topk非常非常慢，而且运行时间和topk的值线性相关
@jax.jit
def gen_spu_greedy(params, input_ids):
    return generate_greedy(model, params, input_ids, n_tokens_to_gen=3) # 贪心采样，最快

@jax.jit
def gen_spu_topk(params, input_ids):
    return generate_topk(model, params, input_ids, n_tokens_to_gen=3, top_k=40) # topk采样，最慢

@jax.jit
def gen_spu_minp(params, input_ids):
    return generate_minp(model, params, input_ids, n_tokens_to_gen=3, min_p=0.1) # minp采样，比贪心慢一点点，但效果很好

# @jax.jit
# def gen_spu(params, input_ids):
#     # return generate(model, params, input_ids, n_tokens_to_gen=3) # 老代码
#     # return generate_greedy(model, params, input_ids, n_tokens_to_gen=3) # 贪心采样，最快
#     # return generate_topk(model, params, input_ids, n_tokens_to_gen=3, top_k=40) # topk采样，最慢
#     return generate_minp(model, params, input_ids, n_tokens_to_gen=3, min_p=0.1) # minp采样，比贪心慢一点点，但效果很好

### 3.2 greedy

In [ ]:
result = emulator.run(gen_spu_greedy)(s_params, s_input_ids)

[2026-04-27 13:33:25,608] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 13:33:31,989] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 13:33:31,994] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 13:33:32,003] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 13:33:32,006] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 13:33:32,008] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 13:33:32,010] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 13:33:32,012] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 13:33:32,015] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 13:33:32,017] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 13:33:32,020] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-27 13:33:32,021] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-27 13:33:32,023] [ForkServerProcess-4

In [ ]:
result = emulator.run(gen_spu_greedy)(s_params, s_input_ids)

In [ ]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

### 3.3 minp

In [ ]:
result = emulator.run(gen_spu_minp)(s_params, s_input_ids)

In [ ]:
result = emulator.run(gen_spu_minp)(s_params, s_input_ids)

In [ ]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

### 3.4 topk

In [ ]:
result = emulator.run(gen_spu_topk)(s_params, s_input_ids)

In [ ]:
result = emulator.run(gen_spu_topk)(s_params, s_input_ids)

In [ ]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

## (4) cleanup

In [ ]:
emulator.down()